# SNS 질문 라벨 검토·수정과 EDA 요약

질문 문맥 검토·수정에 참여한 경험과 팀 EDA 결과를 정리했습니다. 공개 집계 CSV에서 표와 비율을 다시 계산하는 노트북입니다. 원시 로그를 새로 조회하거나 전체 분석을 재실행하지 않습니다.

지표의 단위와 출처는 [SOURCES.md](SOURCES.md)를 참고하세요.

## 1. AI 초기 라벨의 검토·수정

질문이 묻는 핵심 주제를 확인하고 카테고리를 조정했습니다. 아래는 초기 AI 분류 파일과 수정본을 대조한 팀 작업 사례입니다.

In [1]:
from pathlib import Path
import csv

# 저장소 루트, 프로젝트 폴더에서 실행할 수 있습니다.
candidates = [Path.cwd(), Path.cwd().parent,
              Path.cwd() / "projects" / "sns-question-labeling"]
project = next(p for p in candidates if (p / "category_eda_summary.csv").exists())

with (project / "category_eda_summary.csv").open(encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))
for row in rows:
    for key in ["vote_count", "read_count", "initial_open_count", "original_question_id_count"]:
        row[key] = int(row[key])
    row["calculated_read_rate_pct"] = row["read_count"] / row["vote_count"] * 100
    row["calculated_open_relative_to_read_pct"] = row["initial_open_count"] / row["read_count"] * 100
    row["calculated_votes_per_question_id"] = row["vote_count"] / row["original_question_id_count"]
    assert abs(row["calculated_read_rate_pct"] - float(row["read_rate_pct"])) < 0.000001
    assert abs(row["calculated_open_relative_to_read_pct"] - float(row["initial_open_relative_to_read_pct"])) < 0.000001

print(f"카테고리 수: {len(rows)}")
print(f"투표 건수 합계: {sum(r['vote_count'] for r in rows):,}")
print(f"원본 질문 ID 수 합계: {sum(r['original_question_id_count'] for r in rows):,}")


카테고리 수: 8
투표 건수 합계: 1,201,659
원본 질문 ID 수 합계: 4,751


In [2]:
# 노트북에서는 표로 표시하고, 일반 Python에서는 텍스트로 확인합니다.
try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    Markdown = str
    display = print

with (project / "label_revision_examples.csv").open(encoding="utf-8", newline="") as f:
    examples = list(csv.DictReader(f))
lines = ["| 질문 | 초기 라벨 | 수정본 라벨 |", "| --- | --- | --- |"]
for row in examples:
    lines.append(f"| {row['question_text']} | {row['initial_category']} | {row['revised_category']} |")
display(Markdown("\n".join(lines)))


| 질문 | 초기 라벨 | 수정본 라벨 |
| --- | --- | --- |
| 음식 취향이 잘맞는 사람은? | 관계·친밀도 | 생활·취향 |
| 같이 동물원 가고 싶은 사람은? | 연애·이성호감 | 관계·친밀도 |
| 음악 취향이 가장 잘 맞는 친구는? | 관계·친밀도 | 생활·취향 |

## 2. 카테고리별 반응 비교

평균 투표는 원본 질문 ID를 분모로 사용합니다. 읽음 비율과 읽음 대비 초성 열람 비율을 별도로 계산합니다. 초성 열람을 실제 유료 결제로 해석하지 않습니다.

In [3]:
headers = ["카테고리", "원본 질문 ID당 평균 투표", "읽음 비율", "읽음 대비 초성 열람 비율"]
lines = ["| " + " | ".join(headers) + " |", "| --- | --- | --- | --- |"]
for row in sorted(rows, key=lambda r: r["calculated_open_relative_to_read_pct"], reverse=True):
    values = [row["category"], f"{row['calculated_votes_per_question_id']:.1f}회",
              f"{row['calculated_read_rate_pct']:.2f}%",
              f"{row['calculated_open_relative_to_read_pct']:.2f}%"]
    lines.append("| " + " | ".join(values) + " |")
display(Markdown("\n".join(lines)))


| 카테고리 | 원본 질문 ID당 평균 투표 | 읽음 비율 | 읽음 대비 초성 열람 비율 |
| --- | --- | --- | --- |
| 연애·이성호감 | 287.8회 | 58.14% | 13.71% |
| 관계·친밀도 | 218.6회 | 56.78% | 12.32% |
| 외모·비주얼 | 279.8회 | 56.47% | 9.71% |
| 이미지·평판 | 221.8회 | 55.11% | 7.98% |
| 상상·유머 | 234.2회 | 56.32% | 7.90% |
| 성격·기질 | 320.2회 | 54.49% | 6.61% |
| 능력·재능 | 331.7회 | 53.73% | 5.31% |
| 생활·취향 | 185.8회 | 55.17% | 5.12% |

## 3. 해석과 운영 제안

원본 질문 ID당 평균 투표가 높은 카테고리와 읽음 대비 초성 열람 비율이 높은 카테고리가 달랐습니다. 질문의 역할을 비교할 때 참여 규모와 후속 반응을 함께 살펴볼 필요가 있습니다.

팀 발표 자료에서는 중분류별 초성 열람·답장 반응을 토대로 질문 배치와 문항 재설계를 제안했습니다. 이 요약은 관찰된 차이에 대한 EDA와 제안이며, 실제 적용 효과를 검증한 결과로 제시하지 않았습니다.